# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20, 10
2. NaN, 3.0, NaN
3. (2, 2)
4. Series, DataFrame
5. [1, 2] 5

In [106]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [107]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [108]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [109]:
# 1
print(df.shape)
print(df.dtypes)

(5463, 8)
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object


In [110]:
print(df["delivery_days"].isna().sum())

160


In [111]:
print(f"Unique cities: {df["city"].nunique()}")
print(f"Unique categories: {df["category"].nunique()}")
print(f"Unique statuses: {df["status"].nunique()}")
print(f"Unique customer_ids: {df["customer_id"].nunique()}")

# Грязные названия городов (14 вариантов): в столбце city есть пробелы, разный регистр и латиница. Один город может считаться рахынми
# Тестовые заказы: есть 3 строки с customer_id == "TEST", Аномальные данные
# Пропуски: в столбце delivery_days 160 пропусков. Не полные данные.


Unique cities: 14
Unique categories: 5
Unique statuses: 2
Unique customer_ids: 2003


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [112]:
# 2
paid_df = df[paid & not_test]
print(f"Mean: {paid_df['amount'].mean().round(3)}")
print(f"Median: {paid_df['amount'].median()}")
print(f"25-th percentile: {paid_df['amount'].quantile(0.25)}")
print(f"75-th percentile: {paid_df['amount'].quantile(0.75)}")
# Типичный чек около 10 300 (медиана),
# а основной диапазон большинства заказов
# находится в пределах от 5 880 до 18 690.

Mean: 14672.391
Median: 10300.0
25-th percentile: 5880.0
75-th percentile: 18690.0


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [113]:
# 3
Shymkent = ["Шымкент", "шымкент", "Shymkent"]
Almaty = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]

canceled = df["status"] == "отменён"
canceled_total = canceled.mean().round(3)
canceled_sh = canceled[df["city"].isin(Shymkent)].mean().round(3)
canceled_ala = canceled[df["city"].isin(Almaty)].mean().round(3)
print(f"Total canceled: {canceled_total}")
print(f"Canceled in Shymkent: {canceled_sh}")
print(f"Canceled in Almaty: {canceled_ala}")
print(f"Шымкент выше среднего в {canceled_sh / canceled_total:.2f} раза")
print(f"Шымкент выше Алматы в {canceled_sh / canceled_ala:.2f} раза")


Total canceled: 0.102
Canceled in Shymkent: 0.204
Canceled in Almaty: 0.079
Шымкент выше среднего в 2.00 раза
Шымкент выше Алматы в 2.58 раза


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [114]:
# 4
sep = df["order_date"] < "2026-10"
octo = ~sep
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"

SH = ["Шымкент", "шымкент", "Shymkent"]
ALA = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]

In [115]:
base = paid & not_test


In [116]:
rev_sh_sep = df[base & sep & df["city"].isin(SH)]["amount"].sum()
rev_sh_octo = df[base & octo & df["city"].isin(SH)]["amount"].sum()
chg_sh = (rev_sh_octo - rev_sh_sep) / rev_sh_sep * 100

rev_ala_sep = df[base & sep & df["city"].isin(ALA)]["amount"].sum()
rev_ala_octo = df[base & octo & df["city"].isin(ALA)]["amount"].sum()
chg_ala = (rev_ala_octo - rev_ala_sep) / rev_ala_sep * 100

print(f"Revenue in Shymkent in September: {rev_sh_sep.round(3)}")
print(f"Revenue in Shymkent in October: {rev_sh_octo.round(3)}")
print(f"Revenue in Almaty in September: {rev_ala_sep.round(3)}")
print(f"Revenue in Almaty in October: {rev_ala_octo.round(3)}")
print(f"Change in Shymkent: {chg_sh:.2f}%")
print(f"Change in Almaty: {chg_ala:.2f}%")

# Падение сильнее в Шымкенте в 0,183 раза

Revenue in Shymkent in September: 4949800.0
Revenue in Shymkent in October: 4044010.0
Revenue in Almaty in September: 15523940.0
Revenue in Almaty in October: 14643020.0
Change in Shymkent: -18.30%
Change in Almaty: -5.67%


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [117]:
# 5
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [118]:
# 6
customer_counts = df["customer_id"].value_counts()
top_customer = customer_counts.idxmax()
print(f"Top customer: {top_customer}")
top_customer_orders = df.loc[df["customer_id"] == top_customer, ["order_date", "city", "amount", "status"]]
print(len(top_customer_orders), df.loc[df["customer_id"] == top_customer]["order_id"].nunique())
top_customer_orders
# Да есть повторяющиеся order_id, получается мы посчитали один заказ два раза

Top customer: C01818
11 10


,order_date,city,amount,status
57,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,2026-09-24 18:00,Астана,1490.0,оплачен
2617,2026-09-29 09:00,Астана,22170.0,оплачен
2776,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,2026-10-18 08:00,Астана,13600.0,оплачен
4610,2026-10-21 19:00,Алматы,21990.0,оплачен


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [119]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [120]:
# 8